In [ ]:
# %% [markdown]
# ## Aim:
# How to link the rule-based extracted CI_TYPE-GEO pairs with the respective Ci failure impacts
# 
# **Idea**\
# Test using prompt engineering by passing table of CI-GEO pairs to GPT-J model.
# Steps:
# * Load model and apply it always on one chunk of the document to extract CI failure impacts 
# * Use prompt engineering to extract time and location of the CI failure (origin) the CI impacts (impact location)
# or 
# * Pass dataframe of pairs as input to the model
# or
# * Use few shot prompting with example answers
# 
# **Finally:**
# * Compaire all approaches of spatial and temporal linking CI failure impacts

# %%
import os


# # settings for CUDA and PYTORCH
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ["PYTORCH_ALLOC_CONF"]="expandable_segments:True" ## improve memory allocation

# # settings for debugging CUDA errors (pinpoint exact line of error)
os.environ["TORCH_USE_CUDA_DSA"] = "1"
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1" 

# activate global venv explicitly
os.environ["VIRTUAL_ENV"] = "/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/.venv"


# %%
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())  # should give 2
print(torch.cuda.get_device_name())
print(torch.cuda.get_device_properties(0))
# print(torch.cuda.get_device_properties(1))
print(torch.cuda.get_device_capability())
print(torch.cuda.get_arch_list())
print(torch.__version__)
print(torch.version.cuda)


## --> must be CUDA 12.6, torch: 2.91, ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']

import os
import sys
# import subprocess
import re
import time
from glob import glob
from pathlib import Path
import gc
# from io import StringIO
# import json

# from tqdm import tqdm
import numpy as np
import pandas as pd
import geonamescache
# import pyarrow as pa
# import pyarrow.parquet as pq
import langdetect
from langchain_docling import DoclingLoader
import spacy
from huggingface_hub import login
from pdfminer.high_level import extract_text
# from langchain.document_loaders import DirectoryLoader #, UnstructuredLoader
# from langchain_community.document_loaders import DirectoryLoader, UnstructuredLoader
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    AcceleratorOptions,
    AcceleratorDevice,
)
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline
from docling.document_converter import DocumentConverter, FormatOption

# sys.path.append("./")
try:
    from src.settings import settings as s
    import src.document_cleaning as dc
    import src.translation_model as tm
    import src.extraction_model as em
    import src.postprocess as pp
    import src.utils as u
    from submodules.geollama.src.main import GeoLlama
    from submodules.geollama.src.model import TopoModel, RAGModel

except:
    sys.path.append("/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src")
    
    from src.settings import settings as s
    import src.document_cleaning as dc
    import src.translation_model as tm
    import src.extraction_model as em
    import src.postprocess as pp
    import src.utils as u
    from submodules.geollama.src.main import GeoLlama
    from submodules.geollama.src.model import TopoModel, RAGModel




test_mode = True

# login to HF
login(token=os.getenv("HUGGINGFACE_TOKEN")) 
# NOTE raises exception if not env.variable doesnt exist (compared to os.envrion.get and its shortcut os.getenv)


# NOTE. disabled batch size as OOM for CUDA despite chunkwise memory cleaning, nvtop to find best batchsize
BATCH_SIZE = s.BATCH_SIZE  # max for nvidia GPU

# torch.manual_seed(42)

#  automatic linebreaks and multi-line cells.
pd.set_option('display.max_colwidth', 100000)
pd.set_option("display.colheader_justify", "left")

print(os.environ["CUDA_VISIBLE_DEVICES"])

# clean up before applying CUDA
gc.collect()
torch.cuda.empty_cache() 
print(torch.cuda.memory_reserved() / 1e9)
torch.no_grad()

# check if cuda cna be used
# device = transformers.infer_device()
# print(f"Using device: {device}")


# ## TODO test to prevent CUDA-OOM when reused
# ## Source: https://spacy.io/usage/embeddings-transformers
# from thinc.api import set_gpu_allocator, require_gpu

# # Use the GPU, with memory allocations directed via PyTorch.
# # This prevents out-of-memory errors that would otherwise occur from competing
# # memory pools.
# set_gpu_allocator("pytorch")
## require_gpu(0)



# %% [markdown]
# ## Set paths and vars

# %%
# set wd to project root
# os.chdir("/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval")

## set path variables
DOCS_DIR = Path(s.PATH_DATA / "text_sources/")
PARSED_TEXT_DIR = Path(s.PATH_DATA / "parsed_documents/")
NER_PATTERNS_FILEPATH = Path(s.NER_PATTERNS_FILEPATH)
LLM_OUTPUTS_DIR = Path(s.PATH_DATA / "llm_outputs/")
GEOLLM_OUTPUTS_DIR = Path(s.PATH_DATA / "geollm_outputs/")

os.makedirs(PARSED_TEXT_DIR, exist_ok=True)
os.makedirs(s.PATH_LLM_DATA, exist_ok=True)
os.makedirs(GEOLLM_OUTPUTS_DIR, exist_ok=True)


# CI GEO pairs
CI_GEO_FILEPATH = Path( s.PATH_DATA / s.CI_GEO_PAIRS_FILENAME)

## store LLM 1 response and prompt
OUTPUT_LLM1_FILEPATH =  Path(s.PATH_LLM_DATA / s.LLM_DATA_FILENAME)
OUTPUT_GEOLLM_FILEPATH =  Path(s.PATH_LLM_DATA / "geollm_results.csv")




# %% [markdown]
# ### Set test mode

# %%

docs_list_sample = [
        Path(PARSED_TEXT_DIR, "Lloyd's List 2024 - Port of Valencia reopens after devastating floods_cleaned.jsonl"),
        Path(PARSED_TEXT_DIR, "Containerlift 2024 - Valencia Port Resumes Operations Following Devastating Flooding in Spain - Containerlift.co.uk - Transport_Lifting_Shipping_cleaned.jsonl"), 
        Path(PARSED_TEXT_DIR, "ABC 2024 - Traffic jams and flight delays due to heavy rain and lightning storm in Malaga_cleaned.jsonl"),

        Path(PARSED_TEXT_DIR, "Karakatsani 2023 - Greece economy briefing The economic impact of the recent devastating floods in Greece_cleaned.jsonl"),
        Path(PARSED_TEXT_DIR, "Koks 2022 - Brief communication_cleaned.jsonl"),
        Path(PARSED_TEXT_DIR, "European Investment Bank 2025 - Spain_ EIB lends €50 million to Iberdrola to rebuild and climate-proof flood-hit power infrastructure in Valencia_cleaned.jsonl"),
        Path(PARSED_TEXT_DIR, "Wilson 2024 - Flash floods in Spain sweep away cars, disrupt trains and leave several missing _ AP News_cleaned.jsonl"),     
            Path(PARSED_TEXT_DIR, "Wildhagen 2013 - Hochwasser_ Wie die Flut Unternehmen lahmlegt_cleaned.jsonl"),
            Path(PARSED_TEXT_DIR, "AFP 2022 - The_Vibes_Valencia Airport in Madrid briefly shut as lightning hits runway _ World _ The Vibes_cleaned.jsonl"),
            Path(PARSED_TEXT_DIR, "Artemis 2015 - PERILS finalises Storm Desmond UK flood loss estimate at £604m_cleaned.jsonl"),
            Path(PARSED_TEXT_DIR, "Brown 2010 - Economy feels chill as UK grinds to a halt _ The Independent_cleaned.jsonl"),
            # Path(PARSED_TEXT_DIR, "Diakakis 2020 - A systematic assessment of the effects of extreme flash floods on transportation infrastructure and circulation: The example of the 2017 Mandra flood_cleaned.jsonl"),
        Path(PARSED_TEXT_DIR, "EFE 2024 - The DANA storm, live_ The death toll rises to 158_cleaned.jsonl"),
            # Path(PARSED_TEXT_DIR, "Eurelectric 2006 - Impacts of Severe Storms on Electric Grids_cleaned.jsonl"),
            Path(PARSED_TEXT_DIR, "Euronews 2024 - Spain floods_ Death toll rises to 205 as nation braces for more rain _cleaned.jsonl"),
        Path(PARSED_TEXT_DIR, "Ferlita 2023 - Incendi in Sicilia, ecco cosa accade_cleaned.jsonl"),
            Path(PARSED_TEXT_DIR, "Fink 2004 - The 2003 European summer heatwaves and drought - synoptic diagnosis and impacts_cleaned.jsonl"),
        Path(PARSED_TEXT_DIR, "Gilbody Dickerson 2024 - Spain floods_ At least 95 people killed including British man near Malaga _ World News _ Sky News_cleaned.jsonl"),
            # Path(PARSED_TEXT_DIR, "Kadir 2014 - The Impact of Natural Disasters on Critical Infrastructures - A Domino Effect-based Study_cleaned.jsonl"),
            Path(PARSED_TEXT_DIR, "Kaur 2025 - Authorities suspect arson in 17 wildfires across Dalmatian coast, Croatia - The Watchers_cleaned.jsonl"),
            # Path(PARSED_TEXT_DIR, "Keller 2014 - Mapping Natural Hazard Impacts on Road Infrastructure—The Extreme Precipitation in Baden-Württemberg_cleaned.jsonl"),
            Path(PARSED_TEXT_DIR, "Kettle 2020 - Storm Xaver over Europe in December 2013 Overview of energy impacts and North Sea events_cleaned.jsonl"),
            # Path(PARSED_TEXT_DIR, "Koks 2019 - Understanding Business Disruption and Economic Losses Due to Electricity Failures and Flooding_cleaned.jsonl"),
            # Path(PARSED_TEXT_DIR, "Korzilius 2021 Nach der Flut_cleaned.jsonl"),

        Path(PARSED_TEXT_DIR, "Khazai 2013 - Juni-Hochwasser 2013 in Mitteleuropa - Fokus Deutschland Bericht 2 Auswirkungen und Bewältigung_cleaned.jsonl"),
        Path(PARSED_TEXT_DIR, "Rozendaal 2021 - Infrabel_ Flood damage to railway track worth tens of millions of euros _ SpoorPro - incomplete_cleaned.jsonl"),
        Path(PARSED_TEXT_DIR, "Skoulding 2023 - Where are the fires in Italy today as temperatures rise to 47.6C on Sicily_ _ The Independent_cleaned.jsonl"),
        Path(PARSED_TEXT_DIR, "The Guardian 2015 - Storm Desmond damage across Cumbria estimated at £500m _ Storm Desmond_cleaned.jsonl"),

        # long processing
        Path(PARSED_TEXT_DIR, "Treanor 2015 - Storm Desmond damage across Cumbria estimated at £500m _ Storm Desmond _The Guardian_cleaned.jsonl"),
        Path(PARSED_TEXT_DIR, "AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLUVIAS INTENSAS_cleaned.jsonl"),
        
        #     # not part of valid set:
        #     # Path(PARSED_TEXT_DIR, "Krausmann 2014 - STREST report on lessons learned from recent catastrophic events_cleaned.jsonl"), # > 1800 entries LLMv3.0 incl. hallucinations
]


## Test mode
if test_mode:
    search_path = docs_list_sample
    print("Test mode is ON. Using only a small sample of documents for testing.")
else:
    search_path = glob(str(Path(PARSED_TEXT_DIR, "*cleaned.jsonl")))

print(f"Using {len(search_path)} documents for processing.")



# %% [markdown]
# ###  Load spaCy language model

## RELOAD spacy pipeline
nlp = spacy.load("./spacy_model_pipeline")

# add CI_TYPE patterns to spacy nlp model pipeline
config = {"spans_key": None, "annotate_ents": True, "overwrite": False}
## see for more info: https://spacy.io/usage/rule-based-matching#entityruler
## NOTE EntityRuler is hidden inside .add_pipe()
try:
    ruler = nlp.add_pipe("span_ruler", config=config)
    ruler.from_disk(s.NER_PATTERNS_FILEPATH)
except ValueError:
    print("SpanRuler already exists in pipeline.")
    ruler = nlp.get_pipe("span_ruler")
    ruler.from_disk(s.NER_PATTERNS_FILEPATH)


# load NER patterns for CI types and their subgroups (needed for cleaning LLm response - STEP 1 before continuing with STEP 2)
ci_patterns = pd.read_json("./ner_patterns.jsonl/patterns", lines=True)

# %%


# %% [markdown]
# ### GeoLLM pipeline
# 
# 

# %%
## Make sure that still both GPUS are visible

# print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# print(os.environ["CUDA_VISIBLE_DEVICES"])
# !nvidia-smi



topo_model = TopoModel(
    model_name='JoeShingleton/GeoLlama-3.2-3b-toponym',
    # model_name='JoeShingleton/GeoLlama_7b_toponym', 
    prompt_path='./submodules/geollama/data/prompt_templates/prompt_template.txt',
    instruct_path='./submodules/geollama/data/prompt_templates/topo_instruction.txt',
    input_path=None,
    config_path='./submodules/geollama/data/config_files/model_config.json'
)

rag_model = RAGModel(
    model_name='JoeShingleton/GeoLlama-3.2-3b-RAG',
    # model_name='JoeShingleton/GeoLlama_7b_RAG',
    prompt_path='./submodules/geollama/data/prompt_templates/prompt_template.txt',
    instruct_path='./submodules/geollama/data/prompt_templates/rag_instruction.txt',
    input_path='./submodules/geollama/data/prompt_templates/rag_input.txt',
    config_path='./submodules/geollama/data/config_files/model_config.json')

geo_llama = GeoLlama(
    topo_model = topo_model, 
    rag_model = rag_model, 
    translate_model=None
)



0
True
1
NVIDIA A100-PCIE-40GB
_CudaDeviceProperties(name='NVIDIA A100-PCIE-40GB', major=8, minor=0, total_memory=40441MB, multi_processor_count=108, uuid=49cb0d59-8657-8b70-8dd2-c1db7aab24a4, pci_bus_id=131, pci_device_id=0, pci_domain_id=0, L2_cache_size=40MB)
(8, 0)
['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']
2.9.1+cu126
12.6
Running on TUB cluster
Running on TUB Cluster
0
0.0
Test mode is ON. Using only a small sample of documents for testing.
Using 24 documents for processing.
Using flash attention
Using locally saved model


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

NOTE - loading tokenizer not from fine-tuned geollm model, but from its anchestor: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
Using flash attention
Using locally saved model


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

NOTE - loading tokenizer not from fine-tuned geollm model, but from its anchestor: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit


## Document cleaning

In [2]:


print( "Number of documents to process:", len(os.listdir(DOCS_DIR)) )
start_time = time.time()

# convert the different layouts of the pdf files into unified markdown format incl. sub/section titles, tables, caption text etc
for pdf_filename in os.listdir(DOCS_DIR):

    if pdf_filename.endswith(".pdf"):

        md_filename = f"{Path(pdf_filename).stem}.md"

        pdf_filepath = os.path.join(DOCS_DIR, Path(pdf_filename))
        md_filepath = os.path.join(PARSED_TEXT_DIR, Path(md_filename))
        cleaned_jsonl_filepath = md_filepath.replace(".md", "_cleaned.jsonl")

        if os.path.exists(cleaned_jsonl_filepath):
            print(f"Cleaned jsonl file already exists: '{cleaned_jsonl_filepath}'")
            continue

        print(f"\nFetching: {pdf_filename}")

        print("Remove reference section")     
        pdf_text = extract_text(pdf_filepath)
        pdf_text_no_refs = dc.remove_references(pdf_text)

        print("Removing URLs") # LangExtract tries to open these URLs when they occur in the document text
        pdf_text_no_refs_urls = re.sub(r"http\S+", "", pdf_text_no_refs) 

        # FIXME remove workaround of saving pdf as markdown and reading it again as Docling.Document
        with open(md_filepath, "w", encoding="utf-8") as f:
            f.write(pdf_text_no_refs_urls)

        print("Converting Markdown to text...")
        # # FIXME with DocLoader
        loader = DoclingLoader(md_filepath)
        md_doc = loader.load()
        # md_text = converter.convert(md_filepath)  #org, ISSUE: converter splits text not at dots --> create incomplete texts (e.g.. losses the end) and writes it to cleand.md where it seems to be contiuing with new sentence but without "dot"
        
        for i, c in enumerate(md_doc):
            # TODO doc .cleaning needs improvement
            c = c.page_content
            
            #c = c.replace(r"\n", r" ")   # Isssue replaces also multipelinebreas eg before subsection
            c = re.sub(r"([^\s-])\n([^\s-])", r"\1 \2", c) # replace linebreak symbols when they occur just once, with whitespace (two linebreaks - probably new subsection)
            c = c.replace("/\n{2,}/g", "\n")  # remove linebreaks only when they occurred just once, but not for multiple linebreaks (e.g. before subsection)
            # Matches \n not preceded or followed by \n
            # c = re.sub(r"(?<!\n)\n(?!\n)", r"\n", c)  # remove linebreaks only when the yoccured just once, but not for multiple linebreaks (e.g. before subsection)
            c = re.sub(r"\s+", " ", c)  # replace >1 whitespaces with single whitespace
            # c = c.replace(r"\w*- ", "\w*-", c)  # removes any word followed by "-"
            # c = re.sub(r"([^\s-])-\n([^\s-])", r"\1\2", c)  # remove hypen and linebreaks TODO test with koks sentences
            c = re.sub(r"([^\s-])- ([^\s-])", r"\1-\2", c)  # remove hypens in the middle of lines

            # print("--> NEW:" ,c)
            # writing cleaned text back to doc
            md_doc[i].page_content = c

        text_cleaned = md_doc

        print("\nRemoving headers and footers")
        print("FIXME - NOT DOING RM HEAD#FOOTERS - issue: removing should be done with Docling-Loader (not Converter- skips ends of sentences)\n")
        # text_cleaned = dc.remove_headers_footers(md_text)

        ## Translation
        src_language_doc = langdetect.detect(pdf_filename.replace(".pdf", "").lower())  # lower case improves language detection

        if src_language_doc != "en":
            supported_languages = ["fr", "de", "es", "it", "itc", "nl"]
            if src_language_doc not in supported_languages:
                print(f"Unsupported source language: {src_language_doc}. Continue with extraction on original text")
                continue 
            
            print(f"Translating {src_language_doc} --> en")
            # for i, chunk in enumerate(text_cleaned):
            #     text_cleaned[i].page_content = tm.translate_2_english(src_language_doc, chunk.page_content)
            text_cleaned = tm.translate_2_english(src_language_doc, text_cleaned)        


        print(f"Saving parsed and cleaned document as jsonl to: {cleaned_jsonl_filepath}")
        # md_text_cleaned.save_as_markdown(cleaned_md_filepath)
        dc.save_doc_to_jsonl(text_cleaned, cleaned_jsonl_filepath)




end_time = time.time() - start_time
print(f"Parsing and cleaning done. Time elapsed: {end_time:.2f} seconds.")


# visual check of removed items
# TODO make as document_cleaning function: print removed items with largest number of chars first
# ## NOTE. high number of chars == more potentially actual text body

# text_items_removed = sorted(text_items_to_drop_visualization, key=lambda x: -x[0])
# for i in text_items_removed[:50]:
#     print(i) # -->  also subsection titles were removed partly


# %%
question_1 = "Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location, and the type of damage."
question_2 = "Is the location of each affected or damaged critical infrastructure correctly identified?"

# %%




gc.collect()
torch.cuda.empty_cache() 
torch.no_grad()
print(torch.cuda.memory_reserved() / 1e9)




Number of documents to process: 47
Cleaned jsonl file already exists: '/beegfs/scratch/a-buch/_PROJECTS/data/parsed_documents/Koks 2022 - Brief communication_cleaned.jsonl'
Cleaned jsonl file already exists: '/beegfs/scratch/a-buch/_PROJECTS/data/parsed_documents/ABC 2024 - Traffic jams and flight delays due to heavy rain and lightning storm in Malaga_cleaned.jsonl'
Cleaned jsonl file already exists: '/beegfs/scratch/a-buch/_PROJECTS/data/parsed_documents/McGregor 2007 - The social impacts of heat waves_cleaned.jsonl'

Fetching: AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLUVIAS INTENSAS.pdf
Remove reference section


/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/document_cleaning.py:45: UserWarning: Expected one match, but found 0 matches,
                taking the last occurred match for determining the start of the reference section.
  warnings.warn(


No References section found!
Removing URLs
Converting Markdown to text...


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.



Removing headers and footers
FIXME - NOT DOING RM HEAD#FOOTERS - issue: removing should be done with Docling-Loader (not Converter- skips ends of sentences)

Translating es --> en
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/hub
Using device: cuda
Using locally saved model from /beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/hub/Helsinki-NLP/opus-mt-es-en


...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

Saving parsed and cleaned document as jsonl to: /beegfs/scratch/a-buch/_PROJECTS/data/parsed_documents/Wildhagen 2013 - Hochwasser_ Wie die Flut Unternehmen lahmlegt_cleaned.jsonl

Fetching: Wilson 2024 - Flash floods in Spain sweep away cars, disrupt trains and leave several missing _ AP News.pdf
Remove reference section


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/document_cleaning.py:45: UserWarning: Expected one match, but found 0 matches,
                taking the last occurred match for determining the start of the reference section.
  warnings.warn(


No References section found!
Removing URLs
Converting Markdown to text...

Removing headers and footers
FIXME - NOT DOING RM HEAD#FOOTERS - issue: removing should be done with Docling-Loader (not Converter- skips ends of sentences)

Saving parsed and cleaned document as jsonl to: /beegfs/scratch/a-buch/_PROJECTS/data/parsed_documents/Wilson 2024 - Flash floods in Spain sweep away cars, disrupt trains and leave several missing _ AP News_cleaned.jsonl

Fetching: Yamashita 2008 - Analysis  control  and economic impact assessment of major blackout events.pdf
Remove reference section
Removing URLs
Converting Markdown to text...

Removing headers and footers
FIXME - NOT DOING RM HEAD#FOOTERS - issue: removing should be done with Docling-Loader (not Converter- skips ends of sentences)

Saving parsed and cleaned document as jsonl to: /beegfs/scratch/a-buch/_PROJECTS/data/parsed_documents/Yamashita 2008 - Analysis  control  and economic impact assessment of major blackout events_cleaned.jsonl
P

In [ ]:
s_p =  glob(str(Path("/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/*d.jsonl")))
for filepath in s_p:
    print("\n\n", filepath,"\n")
    # for file_no, filename in enumerate(s_p):

    #     no_documents = len(s_p)
    #     filepath = Path(filename)
    #     filename_stem = filepath.stem


    ## load doc

    # loader = DoclingLoader(filepath)  # use chunks from Docling.Loader
    # doc = loader.load()

    doc = dc.load_doc_from_jsonl(filepath)
    for i,c in enumerate(doc):
        print(i,"\n", c.page_content)
#     doc = dc.load_doc_from_jsonl(filepath)


In [ ]:
s_p =  glob(str(Path("/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/*d.jsonl")))
for filepath in s_p:
    print("\n\n", filepath,"\n")
    # for file_no, filename in enumerate(s_p):


    ## load doc

    #loader = DoclingLoader(filepath)  # use chunks from Docling.Loader
    #doc = loader.load()

    doc = dc.load_doc_from_jsonl(filepath)
    for i,c in enumerate(doc):
        print(i,"\n", c.page_content)



##  CI impact extraction


In [ ]:
# %%
# Settings
model_name = "meta-llama/Llama-3.1-8B-Instruct"

time0 = time.time()

print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# print(os.environ["CUDA_VISIBLE_DEVICES"])

# clean up before applying CUDA
gc.collect()
torch.cuda.empty_cache() 
print(torch.cuda.memory_reserved() / 1e9)
torch.no_grad()

## init LLM extraction models
decoder_model_1 = em.DecoderModelCaching(
    model_name,
    em.load_prompt_template(template_filename="short_static_llama3_NER.txt",)
)

decoder_model_2 = em.DecoderModelCaching(
    model_name,
    em.load_prompt_template(template_filename="short_static_llama3_NER_geollm_step2.txt",)
)


gc.collect()
torch.cuda.empty_cache() 
torch.no_grad()
print(torch.cuda.memory_reserved() / 1e9)

# print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# print(os.environ["CUDA_VISIBLE_DEVICES"])

# CI-GEO pairs
geolocs_cache = geonamescache.GeonamesCache()
countries = geolocs_cache.get_countries()
ci_geo_countries = [*u.gen_dict_extract(countries, 'name')]



## init outputs
df_resp_step1_all = pd.DataFrame()
df_resp_step2_all = pd.DataFrame()
responses_error_list = []
df_ci_cases_not_grouped = pd.DataFrame()
df_geollm_response = pd.DataFrame()


if test_mode:
    search_path = docs_list_sample
    print("Test mode is ON. Using only a small sample of documents for testing.")
else:
    search_path = glob(str(Path(PARSED_TEXT_DIR, "*cleaned.jsonl")))




## Start CI impact extraction
for file_no, filename in enumerate(search_path):

    time1 = time.time()

    src_language_nonengl = None 

    no_documents = len(search_path)
    filepath = Path(filename)
    filename_stem = filepath.stem


    ## load doc (from jsonl)
    doc = dc.load_doc_from_jsonl(filepath)
    
    # doc=dc.load_doc_from_jsonl(filepath)
    # for i in doc:
    #     print(i)
    #     print(i.page_content, "\n")



    print(f"\n\n ######## -------- Processing document [{file_no+1}/{no_documents}]: {filepath.name} -------- ######## \n")

    ## extract authors, publication year and title 
    author, year, title = dc.extract_citation_info(filename_stem)
    citation = f"{author} {year}".replace("  ", " ").strip()
    title = title.replace(" - ", "").replace("_cleaned", "").strip()



    # init dfs to store interim results for each doc
    df_resp_step1 = pd.DataFrame(
        columns=[
            "citation_id",
            "chunk_id",
            "infrastructure_type",
            "damage",
            "location",
            "ci_entity",
            "geo_entity",
            "chunk_text"
        ]
    )
    df_resp_step2 = pd.DataFrame(
        columns=[
            "citation_id",
            "chunk_id",
            "infrastructure_type",
            "infrastructure_group",
            "damage",
            "location",
            "ci_entity",
            "geo_entity",
            "coord_potential_locations",
            "chunk_text"
        ]
    )
    
    for chunk_no, chunk in enumerate(doc):

        # init dfs for interim results for each chunk
        ## TODO make as pydantic class with fixed attributes
        df_ci_geo_chunk = pd.DataFrame(
            columns=[
                "citation_id",
                "ci_entity",
                "geo_entity",
                "chunk_text",
                "token_distance",
            ]
        )
    
        df_geollm_response = pd.DataFrame()

    
        print(f"\n\nProcessing chunk no. {chunk_no+1} / {len(doc)} of document: {filepath.name}")

        print("Chunk text: ", chunk.page_content)
     
        ## preprocess  TODO move to document cleaning workflow + dc.funcs
        # NOTE currently done in doc. cleaning (when writign to _cleaned.jsonl)
        # chunk.page_content = chunk.page_content.replace("\n", " ")
        # chunk.page_content = chunk.page_content.replace("- ", "-") # TODO test if ("- ", "") is better


        print("\nGetting geolocations of CI assets ")       
        ## get most likely geolocation for each CI entity based on distance between tokens
        nlp_chunk = nlp(chunk.page_content)
        all_ents = [ent for ent in nlp_chunk.ents]
        ci_type_ents = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE", "FAC"]]


        # check if chunk contains CI_TYPE entities
        if len(ci_type_ents) > 0:

            # iterate over all entities within chunk
            for ent_idx in range(len(all_ents)):
                # when entity is CI_TYPE or FAC (i.e. buidling, airports, highways) do following ...
                if all_ents[ent_idx].label_ in ["CI_TYPE", "FAC"]:
                    ci_idx = ent_idx

                    ## .. calculate distances between CI_TYPE entity and  all GEO entities in chunk based on index position
                    distance_list = []
                    idx_in_chunk = []
                    try:
                        for ent_idx in range(len(all_ents)):

                            # TODO calc distances between CI_TYPE ~ GEO entities based on word numbers and not entities (ie tokens)
                            if all_ents[ent_idx].label_ in ["GPE", "LOC"]:

                                ## check that GPE,LOC are not countries (too coarse info CI-GEO pair)
                                ## of GPE/LOC is country -> proceed with next GPE/LOC 
                                if all_ents[ent_idx].text in ci_geo_countries:
                                    continue

                                geo_idx = ent_idx
                                dist_ent_pair = np.abs(ci_idx - geo_idx)
                                distance_list.append(dist_ent_pair)
                                idx_in_chunk.append((ent_idx))
                                closest_pair_idx = np.argmin(distance_list)  # idx of closest GEO entity
                                distance_closest_pair = distance_list[closest_pair_idx]

                        threshold = 5  # max token distance between CI_TYPE and GEO entity
                        if distance_closest_pair > threshold:
                            # print(
                            #     f""" Token distance is too large between CI_TYPE/FAR and next GEO entity which is of {distance_closest_pair} [token distance] > {threshold} [max. token distance] """
                            # )
                            continue
                        else:
                            pass

                        ## write as dict entry incl chunk_id, ci_entity, geo_entity, distance
                        result_dict = {
                            "citation_id": citation,
                            "chunk_text": chunk.page_content,
                            "ci_entity": all_ents[ci_idx].text,
                            "ci_entity_label": all_ents[ci_idx].label_,
                            "geo_entity": all_ents[idx_in_chunk[closest_pair_idx]].text,
                            "geo_entity_label": all_ents[idx_in_chunk[closest_pair_idx]].label_,
                            "token_distance": distance_closest_pair,
                        }
                        df_ci_geo_chunk = pd.concat(
                            [  df_ci_geo_chunk, pd.DataFrame([result_dict])], ignore_index=True
                        )

                    except (IndexError, NameError) as e:
                        continue
        else:
            print("No CI_TYPE entities found in this chunk. Going to next chunk")
            continue

        
        ## post-process of DF CI-GEO pairs for each chunk
        unique_ci_geo_pairs = df_ci_geo_chunk.drop_duplicates(
            subset=["citation_id", "ci_entity", "geo_entity","chunk_text"])
        print("number of duplicates to remove:", len(df_ci_geo_chunk) - len(unique_ci_geo_pairs))

        df_ci_geo_chunk = df_ci_geo_chunk.drop_duplicates(
            subset=["citation_id", "ci_entity", "geo_entity", "chunk_text"]
            )# .reset_index(drop=True, inplace=True)



        print(f"\n Text-2-Data:")

        ## apply decoder on each chunk in document
        ## TODO replace iteration by loading entire document and use recursive chunking from langchain
        time2 = time.time()


        # clean up before applying CUDA
        gc.collect()
        torch.cuda.empty_cache() 
        torch.no_grad()
        # print(torch.cuda.memory_reserved() / 1e9)
    
        print(f"Starting geoparsing")
        ## Start geoparsing with GeoLLM to verify Location
        # load GeoLLM prompt template
        template_geollm = em.load_prompt_template(template_filename="geollm.txt")
        
        
        # extract locations
        # for d in tqdm(chunk.page_content):
        resp = geo_llama.geoparse(chunk.page_content)
        # TODO add here geoLLm prompt as var
        # TODO check if useful in model.py to set : model.use_checkpointing = True or  model.gradient_checkpointing_enable()
        
        # save results
        df_geollm_resp = pd.DataFrame(resp[0:])
        df_geollm_resp["citation_id"] = citation
        df_geollm_resp["chunk_id"] = chunk_no
        df_geollm_resp["chunk_text"] = chunk.page_content
        
        # maybe empty response
        try:
            print("Removing duplicated locations and countries from GeoLLM response to keep only more specific location info")

            df_geollm_resp["name"] = list(set(df_geollm_resp["name"]))  # rm dublicates
            df_geollm_resp = df_geollm_resp[~df_geollm_resp["name"].isin(ci_geo_countries)]  # rm countries
            print("Toponyms from GEoLLM\n", df_geollm_resp["name"])
        except:
            pass

        # saving all geollm repsonses , even when they cannot processed furthernfor later analysis
        df_geollm_response = pd.concat([df_geollm_response, df_geollm_resp], ignore_index=True)
        

        try:
            locations_per_chunk = df_geollm_resp["name"].to_list()
            lats_per_chunk = df_geollm_resp["latitude"].to_list()
            lons_per_chunk = df_geollm_resp["longitude"].to_list()
            RAGestimated_per_chunk = df_geollm_resp["RAG_estimated"].to_list()
        except:
            print("No locations extracted by GeoLLM for this chunk. Going to next chunk\n", resp[0:])
            continue
        
        # sanity check that only cases with location infos are considered for further processing      
        if len(locations_per_chunk) == 0:
            print("GeoLLm did not find any potential locations found for this chunk. Continue with next chunk")
            continue

        ## Input for LLM 1  -1st round
        if df_ci_geo_chunk.empty:
            context = [
                {
                    "text": chunk.page_content,
                    # "citation": citation,
                    # "title": filename_stem,
                    "entity_recognition": None,
                },
            ]

        else:  # TODO dissolve if else clause by making it in ci_locations: if df_ci_geo.chunk=j, xx, else None
            context = [
                {
                    "text": chunk.page_content,
                    # "citation": citation,
                    # "title": filename_stem,
                    "entity_recognition": df_ci_geo_chunk,
                },
            ]


        # print(os.environ["CUDA_VISIBLE_DEVICES"])
        # os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
        # print(os.environ["CUDA_VISIBLE_DEVICES"])

        # load LLM_1 prompt template
        static_prompt = em.load_prompt_template(template_filename="short_static_llama3_NER.txt")
        dynamic_prompt = em.load_prompt_template(template_filename="short_dynamic_llama3_NER.txt")
        # template_1 = em.load_prompt_template(template_filename="ci_loc_direct_impacts_geollm_step_1.txt")

        # apply LLM 1
        response = decoder_model_1.generate_response(
            question=question_1, context=context, # chunk_id=j
            static_prompt = static_prompt,
            dynamic_prompt=dynamic_prompt,
            max_new_tokens = 2048
        )
        # print(type(response ))
        # print(response)
        
        ## postprocess response
        try:
            try:
                df_resp = pp.postprocess_response(response[0])
            except Exception as e:
            # except (IndexError, ValueError) as e:
                df_resp = pp.postprocess_response(response)
            
            # save LLM response for each chunk  in dataframe for each document
            df_resp["citation_id"] = citation 
            df_resp["chunk_id"] = chunk_no  # add chunk id as identifier
            df_resp["ci_entity"] = df_ci_geo_chunk["ci_entity"] 
            df_resp["geo_entity"] = df_ci_geo_chunk["geo_entity"]
            df_resp["coord_potential_locations"] = str(dict(zip(locations_per_chunk, zip(lats_per_chunk, lons_per_chunk, RAGestimated_per_chunk)))) # INTERIM for verification of lat, lon 
            df_resp["chunk_text"] =  context[0]["text"]  # add (translated) chunk text for tracing back LLM response

            if not len(df_resp):
                print("LLM response is empty. Continue with next chunk.\n Response was:", response)
                continue

            # store reponse for chunk to interim df
            df_resp_step1 = pd.concat([df_resp_step1, df_resp], ignore_index=True)


        except (IndexError, ValueError) as e:
            print(f"Cannot add response: {e},\nFaulty response (before postprocessing):", response)
            # faulty response: e.g.  .., "location": "V" on satellite and online on radar"}, { ...}, {}
            responses_error_list.append({
                "citation_id": citation,
                "chunk_id": chunk_no,
                "response": response,
                "error": str(e)
            })
            print("Continue with next chunk\n")
            continue


        ## group Ci types into subgroups, 
        # TODO make nicer when df is empty
        if df_resp.isna().sum().sum() == 0:
            print("No infrastructure types extracted for this chunk, skip grouping into subgroups. Go to next chunk")
            continue

        df_resp = pp.group_ci_types(df_resp, "infrastructure_type", "infrastructure_group", ci_patterns)
        ## store cases which could not be grouped
        df_ci_cases_not_grouped = pd.concat([df_ci_cases_not_grouped, df_resp[df_resp['infrastructure_group'].isna()]], ignore_index=True)
        ## keep only records which are actually about CI (e.g., not theatre, stadion ..)
        df_resp.dropna(subset=["infrastructure_group"], inplace=True)
    

        # clean up after each chunk
        gc.collect()
        torch.cuda.empty_cache()  # mainly needed after training, small effect when LLM applied only for inference
        torch.no_grad()

        print(f"\n   Processing time for chunk {chunk_no+1} STEP 1: {np.round((time.time() - time2) / 60, 1)} minutes\n")

        print("STEP 2")
        print("KEEPING location info from step 1 for further improvement")
        df_locs_org = df_resp.copy() 


        # clean up before applying CUDA
        gc.collect()
        torch.cuda.empty_cache() 
        print(torch.cuda.memory_reserved() / 1e9)
        torch.no_grad()

        # print(os.environ["CUDA_VISIBLE_DEVICES"])
        # os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
        # print(os.environ["CUDA_VISIBLE_DEVICES"])


        ## Input for LLM - 2nd round (GeoLLM)
    
        # print(f"\nCHECKING input.text, df_resp, GeoLLm resp \n{chunk.page_content}\n{df_resp},\n{locations_per_chunk}")
        static_prompt_geollm = em.load_prompt_template(template_filename="short_static_llama3_NER_geollm_step2.txt")
        dynamic_prompt_geollm = em.load_prompt_template(template_filename="short_dynamic_llama3_NER_geollm_step2.txt")
        
        context = [
            {
                "text": chunk.page_content,
                "citation": citation,
                "title": filename_stem, 
                "previous_response": df_resp,
                "potential_locations": locations_per_chunk,  # list of 1 or multiple location strings (no countries)      
                # "coordinates_potential_locations": list(zip(locations_per_chunk, lats_per_chunk, lons_per_chunk, RAGestimated_per_chunk)),
            },
        ]
        
        ## LLM 1 - step2
        response = decoder_model_2.generate_response(
            question=question_2, context=context, # chunk_id=j
            static_prompt = static_prompt_geollm,
            dynamic_prompt = dynamic_prompt_geollm,
            max_new_tokens = 2048
        )

        ## postprocess response
        try:
            try:
                df_resp_2 = pp.postprocess_response(response[0])
            except Exception as e:
            # except (IndexError, ValueError) as e:
                df_resp_2 = pp.postprocess_response(response)

            # save LLM response for each chunk  in dataframe for each document
            df_resp_2["citation_id"] = citation
            df_resp_2["chunk_id"] = chunk_no  # add chunk id as identifier
            df_resp_2["ci_entity"] = df_ci_geo_chunk["ci_entity"] 
            df_resp_2["geo_entity"] = df_ci_geo_chunk["geo_entity"]
            df_resp_2["coord_potential_locations"] = str(dict(zip(locations_per_chunk, zip(lats_per_chunk, lons_per_chunk, RAGestimated_per_chunk)))) # INTERIM for verification of lat, lon 
            df_resp_2["infrastructure_type_org"] = df_locs_org["infrastructure_type"] 
            df_resp_2["damage_org"] = df_locs_org["damage"] 
            df_resp_2["locations_org"] = df_locs_org["location"] 
            df_resp_2["chunk_text"] =  context[0]["text"]  # add (translated) chunk text for tracing back LLM response

            # collect resps for each doc
            print("CREATED final LLM response (STEP 1 & 2) successfully")
            df_resp_step2 = pd.concat([df_resp_step2, df_resp_2], ignore_index=True)
            # print(df_resp_2)

        except (IndexError, ValueError) as e:
            try: 
                print(f"Cannot add response: {e},\nFaulty response (before postprocessing):", response)
                # faulty response: e.g.  .., "location": "V" on satellite and online on radar"}, { ...}, {}
                responses_error_list.append({
                    "citation_id": citation,
                    "chunk_id": chunk_no,
                    "response": response, #.replace('\n', ''),
                    "error": str(e)
                })
            except:
                pass

        print(f"\n   Processing time for chunk {chunk_no} STEPs 1 & 2: {np.round((time.time() - time2) / 60, 1)} minutes\n")

        # clean up before applying CUDA
        gc.collect()
        torch.cuda.empty_cache() 
        print(torch.cuda.memory_reserved() / 1e9)
        torch.no_grad()

        print(os.environ["CUDA_VISIBLE_DEVICES"])



    print(f"\n   Processing time for document {citation} STEP 1&2: {np.round((time.time() - time1) / 60, 1)} minutes\n")

    print(f"Safety: saving responses (Step 1 + 2) for doc: {citation} ")
    df_resp_step1.to_csv(f"llm1_geollm_step1_{citation}.csv", encoding='utf-8', index=False)
    df_resp_step2.to_csv(f"llm1_geollm_step2_{citation}.csv", encoding='utf-8', index=False)

    # clean up after each document
    gc.collect()
    torch.cuda.empty_cache()  # mainly needed after training, small effect when LLM applied only for inference
    torch.no_grad()

    # collecting all docs
    df_resp_step1_all = pd.concat([df_resp_step1_all, df_resp_step1], ignore_index=True)
    df_resp_step2_all = pd.concat([df_resp_step2_all, df_resp_step2], ignore_index=True)



print(f"\n\n ######## -------- CI impact extraction took {(time.time() - time0) / 60} minutes -------- ######## \n\n")


# %%


# %% [markdown]
# ### Finish run

# %%
print("Where CI types could not be grouped:\n", df_ci_cases_not_grouped)


# %%

df_resp_step2.info()

# %%
print(len(df_resp_step1))
unique_ci_geo_pairs = df_resp_step1.drop_duplicates()
print("number of duplicates to remove:", len(df_resp_step1) - len(unique_ci_geo_pairs))

df_resp_step1_nodupl = df_resp_step1.drop_duplicates( )# .reset_index(drop=True, inplace=True)
print(len(df_resp_step1_nodupl))
df_resp_step1_nodupl

# %%
print(len(df_resp_step2))
unique_ci_geo_pairs = df_resp_step2.drop_duplicates()
print("number of duplicates to remove:", len(df_resp_step2) - len(unique_ci_geo_pairs))

df_resp_step2_nodupl = df_resp_step2.drop_duplicates( )# .reset_index(drop=True, inplace=True)
print(len(df_resp_step2_nodupl))
df_resp_step2_nodupl


print(df_resp_step2.infrastructure_group.isna().sum())  # mostly cases which are not CI (theater, stadion..)
print(df_resp_step2.infrastructure_group.value_counts()) # four most common subgroups seems to be correct
# df_pred.infrastructure_group.unique()



gc.collect()
torch.cuda.empty_cache() 
torch.no_grad()
print(torch.cuda.memory_reserved() / 1e9)
# %%
print(torch.cuda.memory_reserved() / 1e9)


# %%
print("Chunk with erroneous responses:", responses_error_list.__len__())
df_responses_error = pd.DataFrame(responses_error_list)
# df_responses_error#.tail(3)


# %%
# df_responses_all_step2#.tail(3)

# %% [markdown]
# ### Saving

# %%
# PATH_LLM_DATA: Path = Path(s.PATH_DATA /"llm_outputs/")
# LLM_DATA_FILENAME: str = "llm_1_updprompt_distanceNER.csv"

# OUTPUT_LLM1_FILEPATH =  Path(PATH_LLM_DATA / LLM_DATA_FILENAME)#.replace(".csv", "_v2.csv"))
# OUTPUT_LLM1_FILEPATH 

# %%
safety_df = df_resp_step2_all.copy()

# save LLM 1 output to disk along with prompt text
if not os.path.isfile(OUTPUT_LLM1_FILEPATH):

    print(f"Saving prompt, LLM response and erroneous responses [.txt, .csv] to {OUTPUT_LLM1_FILEPATH} ...")

    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_staticgeol_{OUTPUT_LLM1_FILEPATH.stem}.txt", "w") as f:
        f.write(static_prompt.render(context=context, question=question_1))
    try:
        with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_dynamicgeol_{OUTPUT_LLM1_FILEPATH.stem}.txt", "w") as f:
            f.write(dynamic_prompt.render(context=context, question=question_1))
    except Exception as e:
        print("UndefinedError: dynamic_prompt has probably no df_ci_geo (it is empty)")   
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_staticgeol_{OUTPUT_LLM1_FILEPATH.stem}.txt", "w") as f:
        f.write(static_prompt_geollm.render(context=context, question=question_2))
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_dynamicgeol_{OUTPUT_LLM1_FILEPATH.stem}.txt", "w") as f:
        f.write(dynamic_prompt_geollm.render(context=context, question=question_2))

    df_resp_step1_all.to_csv(f"{OUTPUT_LLM1_FILEPATH.stem}_step1.csv", encoding='utf-8', index=False)
    df_resp_step2_all.to_csv(f"{OUTPUT_LLM1_FILEPATH.stem}_step2.csv", encoding='utf-8', index=False)
    df_responses_error.to_csv(OUTPUT_LLM1_FILEPATH.parent / f"errors_{OUTPUT_LLM1_FILEPATH.stem}.csv", encoding='utf-8', index=False)

elif os.path.isfile(OUTPUT_LLM1_FILEPATH) and not os.path.isfile(OUTPUT_LLM1_FILEPATH.parent / f"{OUTPUT_LLM1_FILEPATH.stem}_v2.csv"):

    print(f"Output file {Path(OUTPUT_LLM1_FILEPATH).stem} already exists. Saving as {OUTPUT_LLM1_FILEPATH.stem}_v2 to avoid overwriting ...")

    # If the original files exists but the v2 file doesn't, create the v2 file
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_staticgeol_{OUTPUT_LLM1_FILEPATH.stem}_v2.txt", "w") as f:
        f.write(static_prompt.render(context=context, question=question_1))
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_dynamicgeol_{OUTPUT_LLM1_FILEPATH.stem}_v2.txt", "w") as f:
        f.write(dynamic_prompt.render(context=context, question=question_1))
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_staticgeol_{OUTPUT_LLM1_FILEPATH.stem}_v2.txt", "w") as f:
        f.write(static_prompt_geollm.render(context=context, question=question_2))
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_dynamicgeol_{OUTPUT_LLM1_FILEPATH.stem}_v2.txt", "w") as f:
        f.write(dynamic_prompt_geollm.render(context=context, question=question_2))
    
    df_resp_step1_all.to_csv(Path(OUTPUT_LLM1_FILEPATH.parent, f"{OUTPUT_LLM1_FILEPATH.stem}_step1_v2.csv"), encoding='utf-8', index=False)
    df_resp_step2_all.to_csv(Path(OUTPUT_LLM1_FILEPATH.parent, f"{OUTPUT_LLM1_FILEPATH.stem}_step2_v2.csv"), encoding='utf-8', index=False)
    df_responses_error.to_csv(Path(OUTPUT_LLM1_FILEPATH.parent / f"errors_{OUTPUT_LLM1_FILEPATH.stem}_v2.csv"), encoding='utf-8', index=False)

else:
    print(f"Output file {Path(OUTPUT_LLM1_FILEPATH).stem} already exists. Skip saving to avoid overwriting ...")




0
25.828524032
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/hub
Using flash attention
Using locally saved model from /beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/hub/meta-llama/Llama-3.1-8B-Instruct


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Using iterative caching of prompt key values to avoid recomputing entire prompt for each generation step.
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/hub
Using flash attention
Using locally saved model from /beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/hub/meta-llama/Llama-3.1-8B-Instruct


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Using iterative caching of prompt key values to avoid recomputing entire prompt for each generation step.
28.261220352
Test mode is ON. Using only a small sample of documents for testing.


 ######## -------- Processing document [1/24]: Lloyd's List 2024 - Port of Valencia reopens after devastating floods_cleaned.jsonl -------- ######## 



Processing chunk no. 1 / 3 of document: Lloyd's List 2024 - Port of Valencia reopens after devastating floods_cleaned.jsonl
Chunk text:  12/29/25, 9:51 PM Port of Valencia reopens after devastating ﬂoods :: Lloyd's List Take Lloyd's List on the go. 🌍 Access trusted maritime insights anytime, anywhere. 👉 Download the app now! We use cookies to improve your website experience. To learn about our use of cookies and how you can manage your cookie settings, please see our Cookie Policy. By continuing to use the website, you consent to our use of cookies. This copy is for your personal, non-commercial use. For high-quality copies or electronic reprints fo

...

Starting geoparsing
Removing duplicated locations and countries from GeoLLM response to keep only more specific location info
Toponyms from GEoLLM
 0     Valencia
2    Newcastle
Name: name, dtype: str

   Processing time for chunk 5 STEP 1: 6.1 minutes

STEP 2
KEEPING location info from step 1 for further improvement
28.261220352
CREATED final LLM response (STEP 1 & 2) successfully

   Processing time for chunk 4 STEPs 1 & 2: 6.3 minutes

28.261220352
0


Processing chunk no. 6 / 8 of document: Containerlift 2024 - Valencia Port Resumes Operations Following Devastating Flooding in Spain - Containerlift.co.uk - Transport_Lifting_Shipping_cleaned.jsonl
Chunk text:  India’s Big Shipping Play: How a New Maritime Power is Reshaping Global Trade ( maritime-power-is-reshaping-global-trade/) November 4, 2025 Top Container Carriers Show Widening Reliability GapDate: 3 November 2025 ( reliability-gapdate-3-november-2025/) November 3, 2025 DARKFIELD London 2025: Immersive Sound Theatre Inside Shi

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting geoparsing
Removing duplicated locations and countries from GeoLLM response to keep only more specific location info
Toponyms from GEoLLM
 0     Valencia
3       London
4    Clydeside
Name: name, dtype: str

   Processing time for chunk 6 STEP 1: 0.9 minutes

STEP 2
KEEPING location info from step 1 for further improvement
28.261220352
CREATED final LLM response (STEP 1 & 2) successfully

   Processing time for chunk 5 STEPs 1 & 2: 1.0 minutes

28.261220352
0


Processing chunk no. 7 / 8 of document: Containerlift 2024 - Valencia Port Resumes Operations Following Devastating Flooding in Spain - Containerlift.co.uk - Transport_Lifting_Shipping_cleaned.jsonl
Chunk text:  6/7 1/3/26, 11:52 PM Valencia Port Resumes Operations Following Devastating Flooding in Spain - Containerlift.co.uk - Transport/Lifting/Ship… Resources( Contact us ( Containerlift Services Ltd Gallop House, Haslers Lane Great Dunmow, Essex CM6 1XS, Great Britain +44 (0)1371 879400(tel:01371879400) enquiries@cont

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting geoparsing
Removing duplicated locations and countries from GeoLLM response to keep only more specific location info
Toponyms from GEoLLM
 1    Great Britain
Name: name, dtype: str


## Doc. cleaning improve

In [ ]:
# c = """ 
# blublub title\n\n\n

# HERE IS NEW SUBSECTION:\n

# (large-scale) societal dis- ruptions dis - ruptions (Garschagen and Sandholz, 2018; Hallegatte et al., 2019; Fekete and Sandholz, 2021), empirical evidence on the impacts of extreme weather events on these systems is still

# Published by Copernicus Publications on behalf of the European Geosciences Union.

# E. E. Koks et al.: Flood impacts to infrastructure

# limited. This brief communication provides an overview of the observed ﬂood impacts to large-scale infrastructure sys- tems during the 2021 mid-July western European ﬂood event and how reconstruction of these large-scale systems has pro- 




# HERE IS NEW SUBSECTION

# severely damaged railway line (between the vil- lages of Spa and Pepinster) was reopened again on 3 Octo- ber 2021 (Rozendaal, 2021b). In the Netherlands, no large- scale damage has been reported to transport infrastructure. A few national highways were partly ﬂooded (e.g. the A76 in both directions) or brieﬂy closed (&lt; 3 d) because of the po- tential of ﬂooding. Most likely due to relative low-ﬂow ve- locities, damage to Dutch national road infrastructure was limited. Several railway sections were closed (e.g. the rail-

# way section between Maastricht and Liége) and some dam- age occurred to the railway infrastructure, in particular to the electronic “track circuit” devices and saturated railway em- bankments (Prorail, 2021).

# """
# #c = c.replace(r"\n", r" ")   # Isssue replaces also multipelinebreas eg before subsection
# c = re.sub(r"([^\s-])\n([^\s-])", r"\1 \2", c) # replace linebreak symbols when they occur just once, with whitespace (two linebreaks - probably new subsection)
# c = c.replace("/\n{2,}/g", "\n")  # remove linebreaks only when they occurred just once, but not for multiple linebreaks (e.g. before subsection)
# # Matches \n not preceded or followed by \n
# # c = re.sub(r"(?<!\n)\n(?!\n)", r"\n", c)  # remove linebreaks only when the yoccured just once, but not for multiple linebreaks (e.g. before subsection)
# c = re.sub(r"\s+", " ", c)  # replace >1 whitespaces with single whitespace

# # c = c.replace(r"\w*- ", "\w*-", c)  # removes any word followed by "-"
# # c = re.sub(r"([^\s-])-\n([^\s-])", r"\1\2", c)  # remove hypen and linebreaks TODO test with koks sentences
# c = re.sub(r"([^\s-])- ([^\s-])", r"\1-\2", c)  # remove hypens in the middle of lines

# c 


# # 0090- some weird breaks-
# # And some long sentences 
# # which are not separated by dots but by line breaks and hyphens 



In [ ]:
# !uv pip install "unstructured[pdf]"  # unstructured  #langchain-unstructured #langchain-community
# # # !uv add langchain
# !uv lock
# !uv sync
# from langchain_community.document_loaders import DirectoryLoader, UnstructuredFileLoader
# #from langchain.loaders import , UnstructuredFileLoader

# loader = DirectoryLoader(str(Path(DOCS_DIR)),  loader_cls=UnstructuredFileLoader, show_progress=True)
# pdf_docs = loader.load()
# # glob=glob("*.pdf"),
# print(f"Number of Documents: {len(pdf_docs)}")

# # convert the different layouts of the pdf files into unified markdown format incl. sub/section titles, tables, caption text etc
# for idx, doc in enumerate(pdf_docs, start=0):
#     print(doc)

In [ ]:
# pdf_filepath = Path("Lloyd's List 2024 - Port of Valencia reopens after devastating floods.pdf")
# # Path(DOCS_DIR, "Lloyd's List 2024 - Port of Valencia reopens after devastating floods.pdf")
# print(pdf_filepath)
# print("Remove reference section")

# # setup converter for PDF and markdown
# converter = DocumentConverter(
#     allowed_formats=[InputFormat.PDF, InputFormat.MD],
#     format_options={
#         InputFormat.PDF: FormatOption(
#             pipeline_cls=StandardPdfPipeline,
#             pipeline_options=pipeline_options,
#             backend=PyPdfiumDocumentBackend,
#         ),
#     },
# )
# pdf_text = converter.convert(pdf_filepath)
    
# # loader = DoclingLoader("/beegfs/scratch/a-buch/_PROJECTS/data/text_sources/Lloyd's List 2024 - Port of Valencia reopens after devastating floods.pdf")  # use chunks from Docling.Loader
# # pdf_doc = loader.load()